# **Project 5 | Implementing a Recommendation System on Spark**

## Data Preparation, Exploration, and Preprocessing

The dataset used in this project was obtained from the GroupLens Serendipity 2018 dataset, an extension of the MovieLens dataset containing approximately one million user ratings and enriched movie metadata. After loading the movie and ratings datasets into PySpark, an initial exploration of the data was performed to inspect the schema, data types, and overall data quality. An attempt was made to convert the releaseDate column to a date data type; however, several records contained invalid or inconsistent date formats, resulting in conversion errors. Since the release date was not required for the objectives of this study, it was retained as a string before being removed. The remaining columns were assigned appropriate data types during data loading.
To reduce unnecessary features and focus on attributes relevant to the recommendation models, the columns releaseDate, starring, imdbId, and tmdbId were removed. The movie metadata was then combined with the user ratings using an inner join on movieId, ensuring that only ratings with corresponding movie records were retained. This avoided introducing unmatched records into the dataset and provided a consistent set of movies for subsequent analysis. After joining the datasets, missing values were identified in the title, directedBy, and genres columns. Rather than discarding these records, the missing metadata values were replaced with "Unknown". This approach preserved all available user–movie interactions for the baseline ALS model while ensuring that incomplete metadata would not hinder subsequent experiments.


In [0]:
import math

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructField, FloatType, StringType, IntegerType, StructType
from pyspark import StorageLevel
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import Row


In [0]:
# Let's define the schema for the movies dataset
movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("releaseDate", StringType(), True),
    StructField("directedBy", StringType(), True),
    StructField("starring", StringType(), True),
    StructField("imdbId", IntegerType(), True),
    StructField("tmdbId", IntegerType(), True),
    StructField("genres", StringType(), True)
])

movies = spark.read \
    .option("header", True) \
    .schema(movie_schema) \
    .csv("file:/Workspace/MovieLens project/movies.csv")

# Change the releaseDate column to a date type
# Check missing values
movies.filter(col("releaseDate").isNull()).count()

# Check suspicious values
movies.filter(col("releaseDate") == "0000-00-00").show()

# Check values before 1800 or after 2026 (extract year first)
from pyspark.sql.functions import regexp_extract

movies = movies.withColumn(
    "releaseYear",
    regexp_extract(col("releaseDate"), r"(\d{4})", 1).cast("int")
)

# Check invalid years
movies.filter(col("releaseYear") < 1800).show()

movies.filter(col("releaseYear") > 2026).show()

# Let's define the schema for the training dataset
training_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True)
])

ratings = spark.read \
    .option("header", True) \
    .schema(training_schema) \
    .csv("file:/Workspace/MovieLens project/training.csv")

movies.printSchema()
ratings.printSchema()

+-------+--------------------+-----------+--------------------+--------------------+------+------+--------------------+
|movieId|               title|releaseDate|          directedBy|            starring|imdbId|tmdbId|              genres|
+-------+--------------------+-----------+--------------------+--------------------+------+------+--------------------+
|    291|Poison Ivy II (1996)| 0000-00-00|       Anne Goursaud|Alyssa Milano, Jo...|114151| 18220|      Drama,Thriller|
|    545|        Harem (1985)| 0000-00-00|        Arthur Joffé|Nastassja Kinski,...| 89256| 33367|               Drama|
|    624|Condition Red (Be...| 0000-00-00|     Mika Kaurismäki|Victor Argo, Paul...|112712|340210|Action,Drama,Thri...|
|    888|Land Before Time ...| 0000-00-00|     Roy Allen Smith|Scott McAfee, Can...|113596| 19004|Adventure,Animati...|
|   1064|Aladdin and the K...| 0000-00-00|          Tad Stones|John Rhys-Davies,...|115491| 11238|Animation,Childre...|
|   1710|Man of Her Dreams...| 0000-00-0

In [0]:
print("Movies:", movies.count())
print("Ratings:", ratings.count())

display(movies.limit(5))
display(ratings.limit(5))

Movies: 49174
Ratings: 9997850


movieId,title,releaseDate,directedBy,starring,imdbId,tmdbId,genres,releaseYear
1,Toy Story (1995),1995-11-19,John Lasseter,"Tim Allen, Tom Hanks, Don Rickles, Jim Varney, John Ratzenberger, Wallace Shawn, Laurie Metcalf, John Morris, R. Lee Ermey, Annie Potts",114709,862,"Adventure,Animation,Children,Comedy,Fantasy",1995
2,Jumanji (1995),1995-12-15,Joe Johnston,"Jonathan Hyde, Bradley Pierce, Robin Williams, Kirsten Dunst",113497,8844,"Adventure,Children,Fantasy",1995
3,Grumpier Old Men (1995),1995-01-01,Howard Deutch,"Jack Lemmon, Walter Matthau, Ann-Margret , Sophia Loren",113228,15602,"Comedy,Romance",1995
4,Waiting to Exhale (1995),1996-01-15,Forest Whitaker,"Angela Bassett, Loretta Devine, Whitney Houston, Lela Rochon",114885,31357,"Comedy,Drama,Romance",1996
5,Father of the Bride Part II (1995),1995-12-08,Charles Shyer,"Steve Martin, Martin Short, Diane Keaton, Kimberly Williams, George Newbern, Kieran Culkin",113041,11862,Comedy,1995


userId,movieId,rating
142882,91658,2.5
142882,4344,1.0
142882,45720,2.0
142882,4734,2.0
142882,91542,2.0


In [0]:
movies = movies.select(
    "movieId",
    "title",
    "directedBy",
    "genres"
)

# ratings is the larger table
movie_ratings = ratings.join( 
    movies,
    on="movieId",
    how="inner" # we only want ratings that have matching movies
)

movie_ratings.printSchema()
display(movie_ratings.limit(5))

root
 |-- movieId: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- title: string (nullable = true)
 |-- directedBy: string (nullable = true)
 |-- genres: string (nullable = true)



movieId,userId,rating,title,directedBy,genres
91658,142882,2.5,"Girl with the Dragon Tattoo, The (2011)",David Fincher,"Drama,Thriller"
4344,142882,1.0,Swordfish (2001),Dominic Sena,"Action,Crime,Drama"
45720,142882,2.0,"Devil Wears Prada, The (2006)",David Frankel,"Comedy,Drama"
4734,142882,2.0,Jay and Silent Bob Strike Back (2001),Kevin Smith,"Adventure,Comedy"
91542,142882,2.0,Sherlock Holmes: A Game of Shadows (2011),Guy Ritchie,"Action,Adventure,Comedy,Crime,Mystery,Thriller"


In [0]:
movie_ratings.select(
    [
        F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
        for c in movie_ratings.columns
    ]
).show()

# Inspect
movie_ratings.show(5, truncate=False)

+-------+------+------+-----+----------+------+
|movieId|userId|rating|title|directedBy|genres|
+-------+------+------+-----+----------+------+
|      0|     0|     0|    2|     12997| 12967|
+-------+------+------+-----+----------+------+

+-------+------+------+-----------------------------------------+-------------+----------------------------------------------+
|movieId|userId|rating|title                                    |directedBy   |genres                                        |
+-------+------+------+-----------------------------------------+-------------+----------------------------------------------+
|91658  |142882|2.5   |Girl with the Dragon Tattoo, The (2011)  |David Fincher|Drama,Thriller                                |
|4344   |142882|1.0   |Swordfish (2001)                         |Dominic Sena |Action,Crime,Drama                            |
|45720  |142882|2.0   |Devil Wears Prada, The (2006)            |David Frankel|Comedy,Drama                                 

In [0]:
# Clean metadata
movie_ratings = movie_ratings.fillna({
    "title": "Unknown",
    "directedBy": "Unknown",
    "genres": "Unknown"
})

movie_ratings.select(
    [
        F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
        for c in movie_ratings.columns
    ]
).show()

movie_ratings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("movie_ratings")

+-------+------+------+-----+----------+------+
|movieId|userId|rating|title|directedBy|genres|
+-------+------+------+-----+----------+------+
|      0|     0|     0|    0|         0|     0|
+-------+------+------+-----+----------+------+



In [0]:
# We iteratively filter sparse users and movies to reduce sparsity and
# improve the stability of collaborative filtering models such as ALS.
#
# We keep only users and movies with at least 5 ratings.
# A higher threshold may remove too many long-tail movies, which can
# negatively affect recommendation diversity and novelty.
#
# The filtering is repeated until no more users or movies fall below
# the threshold. This is necessary because removing inactive users can
# cause some movies to have fewer than 5 ratings, and removing those
# movies can then cause some users to fall below the threshold.

while True:

    # Count the current number of ratings.
    # This gives us a stopping condition: if the number of ratings does
    # not change after filtering, the dataset has stabilized.
    old_count = movie_ratings.count()

    # Identify users with at least 5 ratings.
    # groupBy() counts how many ratings each user has, and we keep only
    # users who meet the minimum activity threshold.
    valid_users = (
        movie_ratings
        .groupBy("userId")
        .count()
        .filter(F.col("count") >= 5)
        .select("userId")
    )

    # Identify movies with at least 5 ratings.
    # Movies with fewer than 5 ratings are removed because they provide
    # limited information for collaborative filtering models.
    valid_movies = (
        movie_ratings
        .groupBy("movieId")
        .count()
        .filter(F.col("count") >= 5)
        .select("movieId")
    )

    # Keep only ratings from users and movies that satisfy the threshold.
    # Inner joins remove any records that do not have a matching valid user
    # or valid movie.
    next_df = (
        movie_ratings
        .join(valid_users, on="userId", how="inner")
        .join(valid_movies, on="movieId", how="inner")
    )

    # Count the filtered dataset so we can determine whether another
    # iteration is needed.
    new_count = next_df.count()

    # Stop when no more rows are removed.
    # At this point all remaining users and movies have at least 5 ratings.
    if new_count == old_count:
        movie_ratings = next_df
        break

    # Otherwise, continue filtering with the updated DataFrame.
    movie_ratings = next_df


# Display final dataset statistics after iterative filtering.
print(f"Number of ratings: {movie_ratings.count()}")

print(
    f"Number of users: "
    f"{movie_ratings.select('userId').distinct().count()}"
)

print(
    f"Number of movies: "
    f"{movie_ratings.select('movieId').distinct().count()}"
)

Number of ratings: 9934236
Number of users: 95199
Number of movies: 27559


In [0]:
# movie_ratings = spark.table("movie_ratings")

# display(
#    spark.table("movie_ratings")
# )

In [0]:
print("User rating counts:")
(
    movie_ratings.groupBy("userId")
    .count()
    .describe()
    .show()
)

print("Movie rating counts:")
(
    movie_ratings.groupBy("movieId")
    .count()
    .describe()
    .show()
)

User rating counts:
+-------+------------------+------------------+
|summary|            userId|             count|
+-------+------------------+------------------+
|  count|            104661|            104661|
|   mean|153617.99270024174| 95.52603166413468|
| stddev| 31211.06212792976|214.46974618070578|
|    min|            100000|                 1|
|    max|            206985|             19764|
+-------+------------------+------------------+

Movie rating counts:
+-------+-----------------+------------------+
|summary|          movieId|             count|
+-------+-----------------+------------------+
|  count|            49151|             49151|
|   mean|100619.5436715428|203.41091737706253|
| stddev|58690.67033616322| 1163.066407560957|
|    min|                1|                 1|
|    max|           183335|             42120|
+-------+-----------------+------------------+



**Observation:** Before splitting the data into training and testing sets, it is worth discussing the speed of our data preparation, exploration, and preprocessing. We immediately observed that these tasks were slower when using Spark than when using Pandas. In addition, Pandas required much simpler code while executing operations significantly faster.

This is likely because our dataset contains only about one million records, which is relatively small for Spark. Spark is designed to process much larger datasets and distributed computing workloads, so it incurs additional overhead from creating execution plans and scheduling tasks across processes or machines. For datasets that fit comfortably into memory, this overhead can outweigh Spark's advantages. In contrast, Pandas performs operations directly in memory with minimal overhead, making it more efficient for datasets of this size. Furthermore, debugging and validating data transformation logic is generally more straightforward in Pandas than when working with Spark DataFrames.

For these reasons, we saved the cleaned dataset locally so that it can be reused in future projects. This approach eliminates the need to repeat the data cleaning process, allowing us to avoid repeating the data cleaning process and instead focus on feature engineering, model development, and building an effective recommender system.


## Data Split

In [0]:
# ==========================================================
# STEP 1: Split into Train (80%) and Test (20%)
# ==========================================================

# Create a random ordering of each user's ratings
user_window = Window.partitionBy("userId").orderBy(F.rand(seed=52))

# Number each rating for every user
# Also count how many ratings each user has
df = (
    movie_ratings
    .withColumn("row_num", F.row_number().over(user_window))
    .withColumn("num_ratings", F.count("*").over(Window.partitionBy("userId")))
)

# Compute how many ratings should be placed in the test set
# Hold out roughly 20% of each user's ratings
# BUT keep users with only 1 rating entirely in training
df = df.withColumn(
    "test_size",
    F.when(
        F.col("num_ratings") == 1,
        F.lit(0)  # users with one rating get no test samples
    ).otherwise(
        F.greatest(
            F.lit(1),
            (F.col("num_ratings") * 0.2).cast("int")
        )
    )
)

# First "test_size" ratings become test set
test = df.filter(F.col("row_num") <= F.col("test_size"))

# Remaining ratings become training set
train = df.filter(F.col("row_num") > F.col("test_size"))

In [0]:
# ==========================================================
# STEP 2: Split Train into Final Train (90%) and Validation (10%)
# ==========================================================

# Randomize the ratings again inside each user
train_window = Window.partitionBy("userId").orderBy(F.rand(seed=52))

# Number each rating for every user
train = (
    train
    .withColumn("row_num", F.row_number().over(train_window))
    .withColumn("num_ratings", F.count("*").over(Window.partitionBy("userId")))
)

# Hold out 10% for validation
# Keep single-rating users in training
train = train.withColumn(
    "val_size",
    F.when(
        F.col("num_ratings") == 1,
        F.lit(0)
    ).otherwise(
        F.greatest(
            F.lit(1),
            (F.col("num_ratings") * 0.1).cast("int")
        )
    )
)

# First val_size ratings become validation
val = train.filter(F.col("row_num") <= F.col("val_size"))

# Remaining ratings become the final training set
train_final = train.filter(F.col("row_num") > F.col("val_size"))

In [0]:
helper_columns = [
    "row_num",
    "num_ratings",
    "test_size",
    "val_size"
]

train_final = train_final.drop(*[c for c in helper_columns if c in train_final.columns])
val = val.drop(*[c for c in helper_columns if c in val.columns])
test = test.drop(*[c for c in helper_columns if c in test.columns])

# Cache datasets because they will be reused multiple times
# This prevents Spark from recomputing the splitting logic repeatedly
# train_final.cache()
# val.cache()
# test.cache()

In [0]:
# ==========================================================
# Check dataset sizes
# ==========================================================

print("Train:", train_final.count())
print("Validation:", val.count())
print("Test:", test.count())

print("Total:", train_final.count() + val.count() + test.count())
print("Expected:", movie_ratings.count())

Train: 7254551
Validation: 774499
Test: 1968800
Total: 9997850
Expected: 9997850


In [0]:
# ==========================================================
# Every validation/test user should also exist in training
# ==========================================================

train_users = train_final.select("userId").distinct()

test_missing = (
    test.select("userId")
    .distinct()
    .subtract(train_users)
    .count()
)

val_missing = (
    val.select("userId")
    .distinct()
    .subtract(train_users)
    .count()
)

print(f"Test users missing from training: {test_missing}")
print(f"Validation users missing from training: {val_missing}")

Test users missing from training: 0
Validation users missing from training: 0


In [0]:
# ==========================================================
# Every validation/test movies should also exist in training
# ==========================================================

train_movies = train_final.select("movieId").distinct()

print(
    "Test movies missing from training: ",
    test.select("movieId").distinct().subtract(train_movies).count())
print(
    "Validation movies missing from training: ",
    val.select("movieId").distinct().subtract(train_movies).count())

Test movies missing from training:  2429
Validation movies missing from training:  1000


**Observation:** The dataset was first split into training (80%) and testing (20%) on a per-user basis, ensuring that each user remained represented in the training set while reserving a portion of their ratings for testing whenever possible. This approach allows the model to learn each user's preferences while evaluating its ability to predict ratings that were not observed during training. The training set was then further divided into a final training set (90%) and a validation set (10%) using the same user-based strategy, again ensuring that users with only a single rating remained in the training data. The validation set was used for model selection and hyperparameter tuning, while the independent test set provided an unbiased estimate of the model's final performance. Although this splitting strategy may result in some movies being absent from the final training set, these movies were intentionally not forced back into the training data. Retaining them only in the validation or test sets preserves the realism of the evaluation by allowing the model to encounter unseen or sparsely rated movies, which better reflects real-world recommendation scenarios where new or less-popular content may appear. This approach also avoids artificially inflating the model's performance by ensuring that evaluation is conducted on genuinely unseen data.

## Baseline ALS Matrix Factorization


In [0]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop", # Remove NaN predictions
    nonnegative=True, # Ensure that predicted ratings are non-negative
    implicitPrefs=False, # Explicit ratings
    seed=52
)

An ALS (Alternating Least Squares) model was used for matrix factorization of the user–item rating matrix, with userId, movieId, and rating representing users, items, and explicit feedback, respectively. coldStartStrategy="drop" was applied to exclude predictions for unseen users or items during evaluation, preventing NaN values from affecting RMSE. implicitPrefs=False indicated explicit feedback, and a fixed random seed ensured reproducibility. The model was initially trained with nonnegative=True to constrain latent factors to non-negative values, improving interpretability but reducing flexibility. Although removing this constraint improved validation RMSE, it led to significantly worse test performance, suggesting overfitting to the validation set and poor generalization. Therefore, the non-negativity constraint was retained in the final model to improve stability and generalization.

In [0]:
rank = 40 # number of latent factors (hidden features)
regParam = 0.01 # regularization parameter, help prevent overfitting
maxiter = 20 # maximum number of ALS training iterations

# Train the baseline ALS model using the initial hyperparameters:
als_model = als.setParams(
    rank=rank,
    regParam=regParam,
    maxIter=maxiter
).fit(train_final)

The ALS model was initially trained with a different set of hyperparameters, which were later adjusted through hyperparameter tuning before finalizing a rank of 40 latent factors, a regularization parameter of 0.01, and 20 training iterations. The rank controls the number of latent features used to represent users and items, enabling the model to capture hidden preference patterns. The regularization parameter helps prevent overfitting by penalizing overly complex latent factor values, while the number of iterations determines how many optimization cycles the ALS algorithm performs before convergence.

In [0]:
val_predictions = als_model.transform(val)

# Take a look at predictions
val_predictions.select(
    "userId",
    "movieId",
    "rating",
    "prediction",
).show(10)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|100009| 109487|   4.5| 4.4922056|
|100014|  48780|   5.0| 4.4098425|
|100027|    608|   3.0| 3.0845494|
|100027| 130686|   3.5|  3.510715|
|100031|   1036|   3.0| 2.6596196|
|100031|   8360|   3.0| 2.9006846|
|100031|    920|   2.5| 2.8577175|
|100031|  41566|   2.5| 2.6817992|
|100031|   1270|   2.5| 2.8640022|
|100033|  99114|   5.0|  4.971428|
+------+-------+------+----------+
only showing top 10 rows


After training, the model generated predicted ratings for the validation dataset. These predictions were compared with the true ratings to assess how well the model generalized to previously unseen data.

In [0]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(val_predictions)
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 0.6002


In [0]:
# Simple grid search for tuning ALS hyperparameters

# Candidate values to test
ranks = [20, 30, 40]          # Number of latent factors
regParams = [0.01, 0.05]      # Regularization strengths
maxIters = [15, 20]           # Maximum training iterations

# Initialize variables to keep track of the best model
best_rmse = float("inf")      # Start with an infinitely large RMSE
best_model = None             # Will store the best trained ALS model
best_params = None            # Will store the best hyperparameter combination

# Try every combination of hyperparameters
for rank in ranks:
    for reg in regParams:
        for maxIter in maxIters:

            # Train an ALS model using the current hyperparameters
            model = als.setParams(
                rank=rank,
                regParam=reg,
                maxIter=maxIter
            ).fit(train_final)

            # Generate predictions on the validation set
            predictions = model.transform(val)

            # Calculate the validation RMSE
            rmse = evaluator.evaluate(predictions)

            # Display the current hyperparameters and their RMSE
            print(
                f"rank={rank}, reg={reg},"
                f"maxIter={maxIter}, RMSE={rmse:.4f}"
            )

            # Keep this model if it performs better than the previous best
            if rmse < best_rmse:
                best_rmse = rmse
                best_model = model
                best_params = (rank, reg, maxIter)
                
# Display the best hyperparameters and their validation RMSE
print("\nBest parameters;")
print(best_params)
print(f"Best Validation RMSE: {best_rmse:.4f}")

rank=20, reg=0.01,maxIter=15, RMSE=0.6603
rank=20, reg=0.01,maxIter=20, RMSE=0.6553
rank=20, reg=0.05,maxIter=15, RMSE=0.6842
rank=20, reg=0.05,maxIter=20, RMSE=0.6796
rank=30, reg=0.01,maxIter=15, RMSE=0.6320
rank=30, reg=0.01,maxIter=20, RMSE=0.6258
rank=30, reg=0.05,maxIter=15, RMSE=0.6664
rank=30, reg=0.05,maxIter=20, RMSE=0.6612
rank=40, reg=0.01,maxIter=15, RMSE=0.6059
rank=40, reg=0.01,maxIter=20, RMSE=0.6002
rank=40, reg=0.05,maxIter=15, RMSE=0.6531
rank=40, reg=0.05,maxIter=20, RMSE=0.6480

Best parameters;
(40, 0.01, 20)
Best Validation RMSE: 0.6002


A manual grid search was performed to identify a better combination of hyperparameters. The search evaluated different values of rank (20, 30, and 40), regularization strength (0.01 and 0.05), and the maximum number of training iterations (15 and 20). For each combination, an ALS model was trained using the training dataset and evaluated on the validation dataset using RMSE. The model achieving the lowest validation RMSE was selected as the final model.

The best-performing configuration was rank = 40, regParam = 0.01, and maxIter = 20, achieving a validation RMSE of 0.6002. This indicates that increasing the number of latent factors while using a relatively small amount of regularization produced the most accurate predictions on the validation data among the configurations tested.


In [0]:
# Once the best hyperparameters are selected, evaluate on the test set
test_predictions = best_model.transform(test)

test_rmse = evaluator.evaluate(test_predictions)

print(f"Test RMSE = {test_rmse:.4f}")

Test RMSE = 0.8343


The best-performing model identified during hyperparameter tuning was then evaluated on the independent test dataset. This provides an unbiased estimate of the model's performance on unseen data. The final model achieved a test RMSE of 0.8343.

Although the validation RMSE was lower than the test RMSE, the test evaluation provides a more realistic estimate of how well the recommendation model is expected to generalize to new user-item interactions.


## Evaluation Metrics

In [0]:
def evaluate(model, train_df, test_df, movies_df, k=10, threshold=3.5):

    # =========================
    # 1. RMSE + MAE
    # =========================
    preds = model.transform(test_df).dropna()

    rmse = preds.select(
        ((F.col("prediction") - F.col("rating")) ** 2).alias("se")
    ).agg(
        F.sqrt(F.mean("se")).alias("RMSE")
    ).collect()[0][0]

    mae = preds.select(
        F.abs(F.col("prediction") - F.col("rating")).alias("ae")
    ).agg(F.mean("ae").alias("MAE")).collect()[0][0]

    # =========================
    # 2. TOP-K PREDICTIONS (UC-safe)
    # =========================
    users = train_df.select("userId").distinct().limit(1000)
    items = train_df.select("movieId").distinct()

    seen = train_df.select("userId", "movieId")

    candidates = (
        users.crossJoin(items)
            .join(seen, ["userId", "movieId"], "left_anti")
    )

    all_preds = model.transform(candidates).dropna()

    window = Window.partitionBy("userId").orderBy(F.col("prediction").desc())

    topk = (
        all_preds.withColumn("rank", F.row_number().over(window))
                .filter(F.col("rank") <= k)
                .select("userId", "movieId")
    )

    # =========================
    # 3. PRECISION + RECALL
    # =========================
    relevant = test_df.filter(F.col("rating") >= threshold) \
        .select("userId", "movieId").distinct()

    hits = topk.join(relevant, ["userId", "movieId"])

    user_hits = hits.groupBy("userId").count().withColumnRenamed("count", "hits")
    user_recs = topk.groupBy("userId").count().withColumnRenamed("count", "recs")
    user_rels = relevant.groupBy("userId").count().withColumnRenamed("count", "rels")

    stats = user_recs.join(user_hits, "userId", "left") \
                     .join(user_rels, "userId", "left") \
                     .fillna(0)

    precision = stats.select(
        F.mean(
            F.when(F.col("recs") > 0,
                F.col("hits") / F.col("recs"))
            .otherwise(0)
        ).alias("Precision@K")
    ).collect()[0][0]
    
    recall = stats.select(
        F.mean(
            F.when(F.col("rels") > 0,
                F.col("hits") / F.col("rels"))
            .otherwise(0)
        ).alias("Recall@K")
    ).collect()[0][0]

    # =========================
    # 4. NOVELTY
    # =========================
    item_pop = train_df.groupBy("movieId").count()

    total_users = train_df.select("userId").distinct().count()

    item_pop = item_pop.withColumn(
        "popularity",
        F.col("count") / F.lit(total_users)
    )

    novelty_df = topk.join(item_pop, "movieId", "left") \
        .fillna({"popularity": 1e-6}) \
        .withColumn("nov_score", -F.log2(F.col("popularity")))

    novelty = novelty_df.groupBy("userId") \
        .agg(F.mean("nov_score").alias("nov")) \
        .agg(F.mean("nov").alias("Novelty@K")) \
        .collect()[0][0]

    # =========================
    # 5. DIVERSITY (DirectedBy-based)
    # =========================
    recs_meta = topk.join(
        movies_df.select("movieId", "directedBy"),
        "movieId",
        "left"
    )

    a = recs_meta.alias("a")
    b = recs_meta.alias("b")

    pairs = a.join(b, "userId") \
        .where(F.col("a.movieId") < F.col("b.movieId"))

    sim = pairs.withColumn(
        "sim",
        (F.col("a.directedBy") == F.col("b.directedBy")).cast("int")
    )

    diversity = sim.groupBy("userId") \
        .agg(F.mean("sim").alias("avg_sim")) \
        .select(F.mean(1 - F.col("avg_sim")).alias("Diversity@K")) \
        .collect()[0][0]

    # =========================
    # FINAL OUTPUT
    # =========================
    return {
            "RMSE": rmse,
            "MAE": mae,
            "Precision@K": precision,
            "Recall@K": recall,
            "Novelty": novelty,
            "Diversity": diversity
        }

In [0]:
als_results = evaluate(als_model, train_final, test, movie_ratings, k=10)

df = spark.createDataFrame([
    Row(
        Model="ALS",
        RMSE=round(als_results["RMSE"], 4),
        MAE=round(als_results["MAE"], 4),
        PrecisionK=round(als_results["Precision@K"], 4),
        RecallK=round(als_results["Recall@K"], 4),
    )
])

df.show(truncate=False)

+-----+------+------+----------+-------+
|Model|RMSE  |MAE   |PrecisionK|RecallK|
+-----+------+------+----------+-------+
|ALS  |0.8343|0.6158|4.0E-4    |0.0043 |
+-----+------+------+----------+-------+



The ALS model achieved an RMSE of 0.8343 and an MAE of 0.6158, indicating good predictive accuracy for explicit ratings. However, Precision@10 (0.0004) and Recall@10 (0.0043) were considerably lower. This is expected because the recommendation task is evaluated under a strict Top-K setting, where only movies rated 3.5 or higher are considered relevant. Furthermore, each user has only a small proportion of their ratings held out for testing, resulting in relatively few relevant items per user. Since ALS is optimized to minimize rating prediction error rather than maximize Top-K ranking performance, it is common for ranking metrics to be substantially lower than rating prediction metrics.

To reduce computational cost during development, Top-K recommendations were evaluated on a subset of 1,000 users rather than the entire dataset. This reduced execution time while preserving the evaluation methodology. A full evaluation over all users would provide more stable estimates of the ranking metrics but would require substantially greater computational resources due to the large number of user–item predictions.


## Conclusion
For this project, beyond gaining initial experience with distributed recommender systems, we also began preparing for our final project by working with a larger dataset than the one used previously. This is why the scope of this work is broader than our earlier MovieLens experiment, which only included 100K ratings.

We implemented the system using Apache Spark on the free Community Databricks environment. The dataset used in this iteration contains approximately 1 million user ratings, along with an enriched movie metadata dataset. Due to the increased scale, we observed clear differences in performance and computational behavior compared to running the same workflow in a local environment.

Before building and training any recommendation models, we first performed data preparation, exploration, and preprocessing. During this stage, we immediately noticed that exploratory data analysis (EDA) operations were significantly slower in Spark compared to a local setup for smaller datasets. This overhead is largely due to distributed execution planning and cluster initialization, which are unnecessary in local execution.
However, once the data pipeline was established, the implementation of the recommendation model using Alternating Least Squares (ALS) showed clear benefits from Spark’s distributed computing capabilities. Model training, fitting, and validation were noticeably faster and more scalable than what would be expected in a local environment with comparable data size. This demonstrates Spark’s strength in handling iterative matrix factorization tasks over large datasets.

Hyperparameter tuning, however, remained computationally expensive. Each experiment took approximately 30 minutes to complete, depending on the parameter configuration. Although this is still more efficient than what a local machine could realistically handle at this scale, it highlights that tuning remains one of the most resource-intensive steps in recommender system development.

We also observed that evaluation metrics such as precision, recall, and diversity were relatively slow to compute when applied to the full dataset. Because of this, we had to sample a subset of the data for evaluation, which may have introduced some bias or reduced the representativeness of the final results. This trade-off reflects a common practical limitation in large-scale recommender system workflows.
Overall, this experiment showed a clear division of labor between local and distributed environments: local systems are more efficient for initial exploration and lightweight preprocessing, while Spark becomes valuable for large-scale model training, distributed computation, and hyperparameter optimization.

In conclusion, moving to a distributed platform such as Apache Spark becomes necessary when the dataset grows beyond the capacity of a single machine, typically at the scale of millions of interactions or more, or when the computational workload involves expensive iterative algorithms (such as ALS matrix factorization), repeated hyperparameter tuning, or large-scale evaluation. At that point, memory constraints, runtime limitations, and processing bottlenecks make local execution impractical. Spark becomes essential not only for performance reasons, but also for enabling experimentation that would otherwise be infeasible on a single-node system.



Warning: The ALS model was trained and evaluated using RMSE on a free community Databricks environment. However, Spark MLlib’s built-in Top-N recommendation methods (`recommendForAllUsers()` and `recommendForUserSubset()`) could not be used due to the `UC_COMMAND_NOT_SUPPORTED.WITHOUT_RECOMMENDATION` error in Unity Catalog, which restricts certain higher-order Spark SQL functions. To address this, a manual recommendation approach was implemented using the learned user and item latent factors.
